In [1]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/karachi_events_clean.parquet")

# Shared sources
cites = pd.concat([
    df[["eventid", "scite1"]].rename(columns={"scite1": "cite"}),
    df[["eventid", "scite2"]].rename(columns={"scite2": "cite"}),
    df[["eventid", "scite3"]].rename(columns={"scite3": "cite"}),
]).dropna()
per_cite = cites.groupby("cite")["eventid"].nunique()
shared = per_cite[per_cite > 1].index
print("Distinct citations:", len(per_cite), "| shared by 2+ events:", len(shared))
print("Events sharing a citation:", cites[cites["cite"].isin(shared)]["eventid"].nunique())

Distinct citations: 2966 | shared by 2+ events: 411
Events sharing a citation: 664


In [2]:
# Same-group pairs inside different time windows
known = df[df["actor_known"]].sort_values("date")
for days in (7, 30, 90):
    pairs = 0
    for _, g in known.groupby("gname"):
        d = g["date"].values.astype("datetime64[D]").astype("int64")
        right = np.searchsorted(d, d + days, side="right")
        pairs += int((right - np.arange(len(d)) - 1).sum())
    print(f"Same-group pairs within {days} days:", pairs)

Same-group pairs within 7 days: 236
Same-group pairs within 30 days: 632
Same-group pairs within 90 days: 1446


In [3]:
print(per_cite.describe())
print(per_cite.sort_values(ascending=False).head(8).rename(lambda s: str(s)[:70]).to_string())
print("Citations shared by 2-5 / 6-10 / 11+ events:",
      ((per_cite >= 2) & (per_cite <= 5)).sum(),
      ((per_cite >= 6) & (per_cite <= 10)).sum(),
      (per_cite >= 11).sum())

count    2966.000000
mean        1.194538
std         0.569829
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         7.000000
Name: eventid, dtype: float64
cite
"Pakistan: Pro-ISIL Group Condemns AQIS Spokesman for 'Falsely' Claimi    7
"Coast Guards driver   shot   dead, soldier wounded," Dawn (Pakistan),    6
"One group involved in at least five sectarian attacks, say police," D    6
Lexis Nexis, "Sixth Blast in Pakistan's Karachi," BBC Monitoring South    6
Xinhua News Agency, "One Killed, Scores Injured in Series of Blasts in    6
"Six MQM activists killed in last 24 hours," Daily Messenger, May 16,     5
"Two cops, MQM activist among 11 shot dead," The News International, S    5
"Express Media attack suspects killed in encounter: police," Plus News    5
Citations shared by 2-5 / 6-10 / 11+ events: 406 5 0
